# **VI.  Análisis de variables numéricas**

## Objetivo
Este notebook tiene la función de analizar la distribución, propiedades estadísticas y anomalías de variables numéricas presentes en el dataset oncológico de GRD (2019-2024), permitiendo identificar valores atípicos, nulos, ceros y patrones temporales. El análisis genera visualizaciones (boxplots) y reportes de terciles para evaluar la calidad de datos numéricos.

## Proceso
1. **Configuración inicial**: Especificar rutas de datos y variables numéricas a analizar
2. **Limpieza de datos numéricos**: Conversión de strings a números, manejo de decimales con coma, valores problemáticos
3. **Cálculo de estadísticas**: Media, mediana, desviación estándar, cuantiles, coeficiente de variación
4. **Detección de outliers**: Identificación usando regla IQR (Interquartile Range)
5. **Análisis de distribución**: Clasificación en terciles (BAJO, MEDIO, ALTO)
6. **Generación de visualizaciones**: Boxplots por año para cada variable
7. **Exportación de resultados**: Tablas resumen, gráficos y distribuciones en CSV/PNG

## Descripciones de columnas de salida:

| Columna | Significado |
|---------|-------------|
| **Año** | Año del análisis (2019-2024) |
| **Variable** | Nombre de la variable numérica analizada |
| **Total** | Número total de registros |
| **Nulos** | Cantidad de valores faltantes (NaN, vacíos) |
| **Ceros** | Cantidad de valores exactamente 0 |
| **Min/Max** | Valores mínimo y máximo |
| **Media** | Promedio aritmético |
| **Mediana** | Percentil 50% (valor central) |
| **Std** | Desviación estándar |
| **CV** | Coeficiente de Variación (Std/Media): mide variabilidad relativa |
| **Q1/Q2/Q3** | Cuartiles (25%, 50%, 75%) |
| **IQR** | Rango Intercuartílico (Q3-Q1) |
| **Outliers** | Cantidad de valores atípicos según regla IQR |
| **% Nulos** | Porcentaje de datos faltantes |
| **% Outliers** | Porcentaje de outliers respecto al total |

In [ ]:
# ============================================================================
# CONFIGURACIÓN INICIAL: Imports, Rutas y Setup
# ============================================================================

import pandas as pd # Para manipulación de datos
import numpy as np # Para operaciones numéricas
import os # Para manejo de rutas y archivos
import matplotlib.pyplot as plt # Para visualización de datos

# Ruta de salida para resultados numéricos
ruta_resultados_num = "../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas"
os.makedirs(ruta_resultados_num, exist_ok=True) # Crea la carpeta si no existe

# Ruta de datos oncológicos sin procesar
carpeta = "../../Datos/Datos oncológicos (sin procesar)"

# Lista de archivos a procesar (años 2019-2024)
archivos = [
    "GRD_ONCOLOGIA_2019.csv",
    "GRD_ONCOLOGIA_2020.csv",
    "GRD_ONCOLOGIA_2021.csv",
    "GRD_ONCOLOGIA_2022.csv",
    "GRD_ONCOLOGIA_2023.csv",
    "GRD_ONCOLOGIA_2024.csv"
]

# Mapeo de equivalencias para estandarizar nombres de columnas inconsistentes
mapa_columnas = {
    "ï»¿COD_HOSPITAL": "COD_HOSPITAL",       # Corrige corrupción de encoding (BOM UTF-8)
    "ID_BENEFICIARIO": "CIP_ENCRIPTADO"     # Renombra a estándar de proyecto
}

In [29]:
# ============================================================================
# ANÁLISIS DE VARIABLES NUMÉRICAS: Función Principal
# ============================================================================

def analizar_variable_numerica(carpeta, archivos, mapa_columnas, columna):
    """
    - Descripción: Función que realiza análisis estadístico completo de una variable numérica.
                   Itera sobre datasets anuales (2019-2024), limpia valores numéricos,
                   calcula estadísticas robustas, detecta outliers, genera visualizaciones
                   (boxplots) y exporta distribuciones en terciles.
    
    - Entradas:
        - carpeta (str): Ruta carpeta datos oncológicos
        - archivos (list): Lista de nombres de archivos CSV (2019-2024)
        - mapa_columnas (dict): Diccionario mapeo de nombres inconsistentes
        - columna (str): Nombre de variable numérica a analizar
    
    - Salida: pd.DataFrame: Tabla con métricas estadísticas (1 fila por año)
    """
    resultados = []

    # =====================================================================
    # CREAR SUBCARPETA POR VARIABLE (Organización de salidas)
    # =====================================================================
    carpeta_variable = os.path.join(ruta_resultados_num, columna)
    os.makedirs(carpeta_variable, exist_ok=True)

    # =====================================================================
    # PROCESAR CADA AÑO
    # =====================================================================
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año = archivo[-8:-4]  # Extrae año (ej: "2019")

        # Cargar CSV con estandarización de columnas
        df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)

        # Saltar si la columna no existe en este año
        if columna not in df.columns:
            continue

        # Copiar columna para no modificar original
        col = df[columna].copy()

        # =====================================================================
        # LIMPIEZA: Conversión a Números (Manejo de Decimales con Coma)
        # =====================================================================
        # Convertir a string y remover espacios
        col = col.astype(str).str.strip()

        # Reemplazar coma decimal por punto (formato europeo a estándar)
        col = col.str.replace(",", ".", regex=False)

        # Normalizar casos como ",77" → "0.77" y "," → "0."
        col = col.str.replace(r"^\.", "0.", regex=True)
        col = col.str.replace(r"^,", "0.", regex=True)

        # Convertir a numérico (strings invalidos → NaN)
        col = pd.to_numeric(col, errors="coerce")

        # Contadores iniciales
        total = len(col)
        n_null = col.isna().sum()
        n_ceros = (col == 0).sum()

        # Filtrar valores válidos (no nulos)
        col_clean = col.dropna()

        # =====================================================================
        # CASO: Todo Nulo o Vacío
        # =====================================================================
        if len(col_clean) == 0:
            # Agregar fila con datos faltantes
            resultados.append({
                "Año": int(año),
                "Variable": columna,
                "Total": total,
                "Nulos": total,
                "Ceros": 0,
                "Min": None,
                "Max": None,
                "Media": None,
                "Mediana": None,
                "Std": None,
                "CV": None,
                "Q1": None,
                "Q2": None,
                "Q3": None,
                "IQR": None,
                "Outliers": 0,
                "% Nulos": 100.0,
                "% Outliers": 0.0
            })

            print(f"  ADVERTENCIA: {columna} {año}: Sin datos válidos (todo nulo o inválido)")
            continue

        # =====================================================================
        # ESTADÍSTICAS DESCRIPTIVAS
        # =====================================================================
        minimo = col_clean.min()  # Valor mínimo
        maximo = col_clean.max()  # Valor máximo
        media = col_clean.mean()  # Promedio
        mediana = col_clean.median()  # Percentil 50%
        std = col_clean.std()  # Desviación estándar

        # Cuartiles
        q1 = col_clean.quantile(0.25)  # Percentil 25%
        q2 = col_clean.quantile(0.50)  # Percentil 50% (mediana)
        q3 = col_clean.quantile(0.75)  # Percentil 75%

        # Rango Intercuartílico (diferencia Q3-Q1)
        iqr = q3 - q1

        # Coeficiente de Variación (medida de dispersión relativa)
        cv = std / media if media != 0 else None

        # =====================================================================
        # DETECCIÓN DE OUTLIERS (Regla IQR)
        # =====================================================================
        # Outliers: valores fuera del rango [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
        outliers = col_clean[(col_clean < (q1 - 1.5 * iqr)) | (col_clean > (q3 + 1.5 * iqr))]
        n_outliers = len(outliers)

        # =====================================================================
        # DISTRIBUCIÓN EN TERCILES (Bajo, Medio, Alto)
        # =====================================================================
        # Calcular percentiles 33% y 66%
        terciles = col_clean.quantile([0.33, 0.66]).values

        # Validar que los bins sean únicos (caso degenerado)
        if len(set(terciles)) < 2:
            # Si todos los valores son iguales
            categorias = pd.Series(["UNICO"] * len(col_clean), index=col_clean.index)
            dist_terciles = categorias.value_counts()
        else:
            # Clasificar en 3 grupos: BAJO, MEDIO, ALTO
            categorias = pd.cut(
                col_clean,
                bins=[-np.inf, terciles[0], terciles[1], np.inf],
                labels=["BAJO", "MEDIO", "ALTO"],
                duplicates="drop"  # Evita error si hay valores coincidentes
            )
            dist_terciles = categorias.value_counts()

        # =====================================================================
        # AGREGAR FILA A RESULTADOS
        # =====================================================================
        resultados.append({
            "Año": int(año),
            "Variable": columna,
            "Total": total,
            "Nulos": n_null,
            "Ceros": n_ceros,
            "Min": minimo,
            "Max": maximo,
            "Media": round(media, 2),
            "Mediana": round(mediana, 2),
            "Std": round(std, 2),
            "CV": round(cv, 3) if cv else None,
            "Q1": round(q1, 2),
            "Q2": round(q2, 2),
            "Q3": round(q3, 2),
            "IQR": round(iqr, 2),
            "Outliers": n_outliers,
            "% Nulos": round((n_null / total) * 100, 2),
            "% Outliers": round((n_outliers / total) * 100, 2)
        })

        # =====================================================================
        # GENERAR BOXPLOT (Visualización de Distribución y Outliers)
        # =====================================================================
        plt.figure(figsize=(8, 5))
        plt.boxplot(col_clean.dropna())
        plt.title(f"Boxplot: {columna} ({año})")
        plt.ylabel(columna)
        plt.grid(True, alpha=0.3)

        # Guardar gráfico en PNG
        ruta_plot = os.path.join(
            carpeta_variable,
            f"{columna.lower()}_{año}_boxplot.png"
        )

        plt.savefig(ruta_plot, dpi=100, bbox_inches='tight')
        plt.close()

        # =====================================================================
        # GUARDAR DISTRIBUCIÓN EN TERCILES (CSV)
        # =====================================================================
        dist_terciles.to_csv(
            os.path.join(
                carpeta_variable,
                f"{columna.lower()}_{año}_terciles.csv"
            ),
            encoding="utf-8-sig"
        )

        print(f"  ✓ {columna} {año}: Procesado | Outliers: {n_outliers} ({round((n_outliers/total)*100, 2)}%)")

    # =====================================================================
    # TABLA FINAL: Consolidar todos los años
    # =====================================================================
    df_resultado = pd.DataFrame(resultados).sort_values("Año")

    # Guardar resumen en CSV
    ruta_salida = os.path.join(
        carpeta_variable,
        f"{columna.lower()}_resumen_numerico.csv"
    )

    df_resultado.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    print(f"\n  Resumen guardado en: {ruta_salida}")

    return df_resultado

## **1. Variable numérica objetivo:**

### **Consumo de recursos o peso relativo GRD(IR_29301_PESO):** Es la expresión numérica continua que refleja la intensidad del consumo de recursos de un episodio hospitalario en relación con un caso promedio (donde 1.0 es el promedio estándar).
- **Tipo de variable**: Variable objetivo (target).

**Análisis de los datos y visualizaciones (boxplots)**:
- **Distribución fuertemente asimétrica (sesgo positivo):** Los boxplots de 2019 a 2024 muestran que la inmensa mayoría de los episodios se agrupan en la parte baja del gráfico. La mediana oscila de forma muy estable entre 1.05 y 1.09, lo que indica que el 50% de los pacientes oncológicos consume recursos muy cercanos al estándar del sistema.
- **Cola pesada en los outliers (casos de alta complejidad):** La tabla indica que hay entre un 9.9% y un 12.2% de valores atípicos (outliers). En los boxplots, esto se visualiza claramente como una larga línea de puntos negros que se extiende desde el límite superior (cerca de 2.0) hasta el valor máximo de 14.24. Esto probablemente representa a la subpoblación de pacientes críticos con estadías prolongadas, múltiples cirugías y uso intensivo de camas UCI.
- **Calidad de los datos:** Esta variable es bastante completa, tiene 0% de nulos en la mayoría de años y un máximo de 2 nulos en 2024 sobre más de 55.000 registros.
- **Balance de clases:** Los recuentos que se obtuvieron al categorizar la variable en BAJO, MEDIO y ALTO demuestran el éxito de la estrategia metodológica de usar cuantiles (q-cut), logrando que las tres clases queden estadísticamente balanceadas (por ejemplo, en 2024: Alto=18.549, Bajo=18.515, Medio=18.045).

In [ ]:
# ============================================================================
# 1. Análisis de PESO (Variable IR_29301_PESO)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: IR_29301_PESO (Consumo de recursos del paciente)")
print("="*100 + "\n")

analizar_variable_numerica(carpeta, archivos, mapa_columnas, "IR_29301_PESO")


ANALIZANDO VARIABLE: IR_29301_PESO (Consumo de recursos del paciente)

  ✓ IR_29301_PESO 2019: Procesado | Outliers: 4588 (9.92%)
  ✓ IR_29301_PESO 2020: Procesado | Outliers: 4337 (12.29%)
  ✓ IR_29301_PESO 2021: Procesado | Outliers: 4581 (11.8%)
  ✓ IR_29301_PESO 2022: Procesado | Outliers: 5238 (11.68%)
  ✓ IR_29301_PESO 2023: Procesado | Outliers: 6048 (11.68%)
  ✓ IR_29301_PESO 2024: Procesado | Outliers: 6390 (11.59%)

  Resumen guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\IR_29301_PESO\ir_29301_peso_resumen_numerico.csv


,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,IR_29301_PESO,46241,0,11,0.0,14.2492,1.29,1.05,1.13,0.876,0.76,1.05,1.32,0.56,4588,0.0,9.92
1,2020,IR_29301_PESO,35294,0,3,0.0,14.2492,1.41,1.09,1.24,0.876,0.86,1.09,1.39,0.53,4337,0.0,12.29
2,2021,IR_29301_PESO,38836,0,16,0.0,14.2492,1.42,1.09,1.28,0.903,0.85,1.09,1.42,0.57,4581,0.0,11.80
3,2022,IR_29301_PESO,44827,1,43,0.0,14.2492,1.40,1.09,1.28,0.914,0.81,1.09,1.39,0.58,5238,0.0,11.68
4,2023,IR_29301_PESO,51784,0,69,0.0,14.2492,1.40,1.09,1.26,0.903,0.79,1.09,1.39,0.60,6048,0.0,11.68
5,2024,IR_29301_PESO,55111,2,65,0.0,14.2492,1.41,1.09,1.28,0.904,0.79,1.09,1.42,0.64,6390,0.0,11.59


## **2. Resto de variables numéricas:**

### **Pesos en recién nacidos (PESORN1 hasta PESORN4):** Registra el peso al nacer (generalmente en gramos) de los recién nacidos (hasta un cuarto bebé en caso de partos múltiples) producto de un episodio de hospitalización que involucró un parto.
- **Tipo de variable**: Hospitalaria (resultados neonatales).
- **Veredicto**: Descartar completamente, porque al igual que se concluyó con las variables categóricas de condición de alta neonatal o sexo del recién nacido, la propuesta está enfocada exclusivamente en episodios cuyo diagnóstico principal es una neoplasia maligna (C00-C97). Un ingreso hospitalario donde una paciente oncológica da a luz es un evento estadísticamente marginal (un outlier clínico a nivel poblacional), de miles de episodios anuales, solo un poco tiene datos en PESORN1, y variables como PESORN3 o PESORN4 están literalmente 100% vacías en todos los años. Por ende, es una variable con muchos nulos, PESORN1 tiene entre un 99.98% y un 99.99% de valores nulos, PESORN2, PESORN3 y PESORN4 tienen un 100% de nulos (salvo un único caso aislado en 2023 para el RN2), lo cual puede introducir ruido y consumir memoria de forma innecesaria.

In [32]:
# ============================================================================
# 2. Análisis de PESORN1 (Consumo de recursos del Primer Recién Nacido)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: PESORN1 (Consumo de recursos del primer recién nacido)")
print("="*100 + "\n")
analizar_variable_numerica(carpeta, archivos, mapa_columnas, "PESORN1")


ANALIZANDO VARIABLE: PESORN1 (Consumo de recursos del primer recién nacido)

  ✓ PESORN1 2019: Procesado | Outliers: 0 (0.0%)
  ✓ PESORN1 2020: Procesado | Outliers: 2 (0.01%)
  ✓ PESORN1 2021: Procesado | Outliers: 1 (0.0%)
  ✓ PESORN1 2022: Procesado | Outliers: 2 (0.0%)
  ✓ PESORN1 2023: Procesado | Outliers: 0 (0.0%)
  ✓ PESORN1 2024: Procesado | Outliers: 0 (0.0%)

  Resumen guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\PESORN1\pesorn1_resumen_numerico.csv


,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN1,46241,46237,0,550.0,3570.0,2582.50,3105.0,1398.81,0.542,2218.75,3105.0,3468.75,1250.00,0,99.99,0.00
1,2020,PESORN1,35294,35286,1,0.0,4700.0,3368.75,3685.0,1417.45,0.421,3515.00,3685.0,3840.00,325.00,2,99.98,0.01
2,2021,PESORN1,38836,38832,0,1690.0,3870.0,3050.00,3320.0,947.28,0.311,2830.00,3320.0,3540.00,710.00,1,99.99,0.00
3,2022,PESORN1,44827,44821,0,3105.0,4600.0,3522.50,3380.0,539.53,0.153,3292.50,3380.0,3400.00,107.50,2,99.99,0.00
4,2023,PESORN1,51784,51780,0,1715.0,3410.0,2531.25,2500.0,699.25,0.276,2217.50,2500.0,2813.75,596.25,0,99.99,0.00
5,2024,PESORN1,55111,55104,0,1970.0,3875.0,2966.00,2810.0,696.52,0.235,2540.00,2810.0,3513.50,973.50,0,99.99,0.00


In [33]:
# ============================================================================
# 3. Análisis de PESORN2 (Consumo de recursos delSegundo Recién Nacido)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: PESORN2 (Consumo de recursos del segundo recién nacido)")
print("="*100 + "\n")
analizar_variable_numerica(carpeta, archivos, mapa_columnas, "PESORN2")


ANALIZANDO VARIABLE: PESORN2 (Consumo de recursos del segundo recién nacido)

  ADVERTENCIA: PESORN2 2019: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN2 2020: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN2 2021: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN2 2022: Sin datos válidos (todo nulo o inválido)
  ✓ PESORN2 2023: Procesado | Outliers: 0 (0.0%)
  ADVERTENCIA: PESORN2 2024: Sin datos válidos (todo nulo o inválido)

  Resumen guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\PESORN2\pesorn2_resumen_numerico.csv


,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN2,46241,46241,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,100.0,0.0
1,2020,PESORN2,35294,35294,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,100.0,0.0
2,2021,PESORN2,38836,38836,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,100.0,0.0
3,2022,PESORN2,44827,44827,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,100.0,0.0
4,2023,PESORN2,51784,51783,0,3100.0,3100.0,3100.0,3100.0,NaN,NaN,3100.0,3100.0,3100.0,0.0,0,100.0,0.0
5,2024,PESORN2,55111,55111,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,100.0,0.0


In [34]:
# ============================================================================
# 4. Análisis de PESORN3 (Consumo de recursos del Tercer Recién Nacido)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: PESORN3 (Consumo de recursos del tercer recién nacido)")
print("="*100 + "\n")
analizar_variable_numerica(carpeta, archivos, mapa_columnas, "PESORN3")


ANALIZANDO VARIABLE: PESORN3 (Consumo de recursos del tercer recién nacido)

  ADVERTENCIA: PESORN3 2019: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN3 2020: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN3 2021: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN3 2022: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN3 2023: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN3 2024: Sin datos válidos (todo nulo o inválido)

  Resumen guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\PESORN3\pesorn3_resumen_numerico.csv


,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN3,46241,46241,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
1,2020,PESORN3,35294,35294,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
2,2021,PESORN3,38836,38836,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
3,2022,PESORN3,44827,44827,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
4,2023,PESORN3,51784,51784,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
5,2024,PESORN3,55111,55111,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0


In [35]:
# ============================================================================
# 5. Análisis de PESORN4 (Consumo de recursos del cuarto recién nacido)
# ============================================================================

print("\n" + "="*100)
print("ANALIZANDO VARIABLE: PESORN4 (Consumo de recursos del cuarto recién nacido)")
print("="*100 + "\n")
analizar_variable_numerica(carpeta, archivos, mapa_columnas, "PESORN4")


ANALIZANDO VARIABLE: PESORN4 (Consumo de recursos del cuarto recién nacido)

  ADVERTENCIA: PESORN4 2019: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN4 2020: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN4 2021: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN4 2022: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN4 2023: Sin datos válidos (todo nulo o inválido)
  ADVERTENCIA: PESORN4 2024: Sin datos válidos (todo nulo o inválido)

  Resumen guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de variables numéricas\PESORN4\pesorn4_resumen_numerico.csv


,Año,Variable,Total,Nulos,Ceros,Min,Max,Media,Mediana,Std,CV,Q1,Q2,Q3,IQR,Outliers,% Nulos,% Outliers
0,2019,PESORN4,46241,46241,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
1,2020,PESORN4,35294,35294,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
2,2021,PESORN4,38836,38836,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
3,2022,PESORN4,44827,44827,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
4,2023,PESORN4,51784,51784,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
5,2024,PESORN4,55111,55111,0,None,None,None,None,None,None,None,None,None,None,0,100.0,0.0
